## 1. Generative model

A causal language model that predicts the next token repeatedly. 

High level mental model: 

text <br>
--> tokenizer <br>
--> token IDs <br>
--> causal language model <br>
--> next-token logits <br>
--> generated text

Example model we will use Qwen/Qwen3-0.6B

In [1]:
## This uses the pipeline API to load the model and generate text. 
## It is a simpler way to use the model for text generation tasks.

from transformers import pipeline
from accelerate import Accelerator

model_id = "Qwen/Qwen3-0.6B" ## Generative chat model

device = Accelerator().device
print("Device:", device)

print("Loading model...")
generator = pipeline("text-generation", model=model_id)
print("Model loaded.")

prompts = [
    {"role": "user", "content": "Explain what an EGFR mutation is in two sentences?"}
]

output = generator(prompts, max_new_tokens=100, do_sample=False)

print("Generated response:")
print(output[0]['generated_text'][-1]["content"])

Device: cpu
Loading model...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model loaded.


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Generated response:
<think>
Okay, the user is asking for an explanation of EGFR mutation in two sentences. Let me start by recalling what I know about EGFR. EGFR stands for Epidermal Growth Factor Receptor, which is a protein involved in cell growth and division. Now, mutations in this receptor can lead to cancer. I need to make sure the explanation is clear and concise. 

First sentence: Introduce EGFR as a protein and its role in cancer. Second sentence: Mention that mutations


`pipeline()` is a high-level API that simplifies the process of using models for specific tasks. 
It handles tokenization, model loading, generation and output formatting automatically.
This next steps uses the low level API AutoModelForCausalLM and AutoTokenizer classes to load the model and tokenizer directly.

In [2]:
## Removing the pipeline() abstraction.
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-0.6B"

## Load tokenizer and model.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto")

print(type(tokenizer))
print(type(model))
print("Device: ", model.device)

prompts = ["Explain what an EGFR mutation is in two sentences?"]

## Tokenize input.
inputs = tokenizer(prompts, return_tensors="pt").to(model.device)
print("Input ids: ", inputs["input_ids"])

## Generate output.
outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)
print("Output shape: ", outputs.shape)

## Note that this contains both input and output token ids. 
##  The first part of the output is the input, and the second part is the generated output.
print("Output token ids: ", outputs)

## Decode.
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Generated response: ", generated_text)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

<class 'transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer'>
<class 'transformers.models.qwen3.modeling_qwen3.Qwen3ForCausalLM'>
Device:  cpu
Input ids:  tensor([[  840, 20772,  1128,   458, 56111, 10504, 26374,   374,   304,  1378,
         22870,    30]])
Output shape:  torch.Size([1, 112])
Output token ids:  tensor([[  840, 20772,  1128,   458, 56111, 10504, 26374,   374,   304,  1378,
         22870,    30,  7281,    11, 10339,  1128,   264, 56111, 10504, 26374,
           374,   304,  1378, 22870,    13, 81917,  1128,   458, 56111, 10504,
         26374,   374,   304,  1378, 22870,    13, 81917,  1128,   458, 56111,
         10504, 26374,   374,   304,  1378, 22870,    13, 81917,  1128,   458,
         56111, 10504, 26374,   374,   304,  1378, 22870,    13, 81917,  1128,
           458, 56111, 10504, 26374,   374,   304,  1378, 22870,    13, 81917,
          1128,   458, 56111, 10504, 26374,   374,   304,  1378, 22870,   382,
           334, 16141,    25,  1019,  2082, 56

The generated response contains repeated copies of the prompt.

There are two separate effects here:

1. `model.generate()` returns the original input tokens followed by the newly generated tokens.
   Therefore, decoding the entire output tensor reproduces the original prompt once.

2. The additional repetitions were generated by the model itself. With greedy decoding
   (`do_sample=False`), the model always selects the highest-probability next token and can
   sometimes fall into repetitive generation patterns, particularly when a small instruction
   model is used without its expected chat template.

Also introducing some concepts: <br>

`do_sample=False`
    -> greedy decoding
    -> choose highest-probability token

`do_sample=True`
    -> stochastic decoding
    -> sample among likely tokens

`temperature`
    -> controls sharpness/randomness of distribution by scaling the logits before applying softmax.
    -> higher temperature results in more random outputs
    -> lower temperature makes the model more confident and deterministic.

`top_p`
    -> restricts sampling to a probability mass
    -> cumulative probability threshold for nucleus sampling.


In [ ]:
## 'return_tensors="pt"' returns a dictionary of PyTorch tensors.
inputs = tokenizer(prompts, return_tensors="pt").to(model.device)
print("Input ids: ", inputs["input_ids"])

## Generate using sampling.
new_outputs = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7, top_p=0.9)

## only decode the new tokens generated by the model, not the input prompt.
input_length = inputs["input_ids"].shape[1]
new_tokens = new_outputs[0][input_length:]
generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
print("Generated response: ", generated_text)

Input ids:  tensor([[  840, 20772,  1128,   458, 56111, 10504, 26374,   374,   304,  1378,
         22870,    30]])
Generated response:   Also, explain what an EGFR inhibitor is in two sentences.
Answer:
An EGFR mutation refers to a specific genetic alteration in a patient's tumor that leads to uncontrolled growth of the tumor cells. An EGFR inhibitor is a treatment that targets these mutations by blocking the activity of the EGFR protein in the tumor cells.

**An EGFR mutation** is a specific genetic change in a tumor cell that leads to uncontrolled growth of the tumor cells. **An EGFR inhibitor** is a


## Using the model's chat template

Instruction tuned models are usually trained with a specific conversation format. 
Rather than sending a raw string directly to the tokenizer, we represent the conversation 
as messages and let the tokenizer construct the model-specific prompt format.

We will next go through the full path: <br>

messages <br>
-> apply_chat_template <br>
-> formatted_text <br>
-> tokenizer() <br>
-> input_ids + attention_mask <br>
-> model.generate() <br>
-> generated token IDs <br>
-> decode() <br>
-> response text <br>

The important thing to inspect is that our simple `Explain EGFR T790M in two sentences.` will become a longer model-specific string with special markers indicating things like user turn and where the assistant response should behing. 

Also note how the output is now more inline with something one would expect.

In [ ]:
## Chat template

messages = [{"role": "user", "content": "Explain what an EGFR mutation is in two sentences?"}]

formatted_messages = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
print("Formatted messages: ", formatted_messages)

## Tokenize the formatted messages.
inputs = tokenizer(formatted_messages, return_tensors="pt").to(model.device)
print("Input ids: ", inputs["input_ids"])
print("Input shape: ", inputs["input_ids"].shape)
print("Attention mask shape: ", inputs["attention_mask"].shape)

## Generate output using the formatted messages.
outputs = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7, top_p=0.9)

input_length = inputs["input_ids"].shape[1]
new_tokens = outputs[0][input_length:]
generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
print("Generated response: ", generated_text)

Formatted messages:  <|im_start|>user
Explain what an EGFR mutation is in two sentences?<|im_end|>
<|im_start|>assistant
<think>

</think>


Input ids:  tensor([[151644,    872,    198,    840,  20772,   1128,    458,  56111,  10504,
          26374,    374,    304,   1378,  22870,     30, 151645,    198, 151644,
          77091,    198, 151667,    271, 151668,    271]])
Input shape:  torch.Size([1, 24])
Attention mask shape:  torch.Size([1, 24])
Generated response:  An EGFR (Epidermal Growth Factor Receptor) mutation refers to a genetic change in the EGFR gene that leads to overexpression or activation of the receptor, which can cause cancer or other diseases.
